# Cómo usar `dataset_analitico.parquet` y `zonas_bogota.geojson`

SYNC-1 · Integrante 1 (Datos) → Integrantes 2 (predictivo/API) y 3 (clustering).

Los dos archivos se generan con un solo comando:

```bash
python data-engineering/build_dataset.py
```

**Regla de oro:** cruzar siempre por `cod_localidad` (string de 2 dígitos, `01`–`20`),
nunca por `localidad_nombre` (es solo referencia).

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd().parent.parent if (Path.cwd().name == 'notebooks') else Path.cwd()
PROC = ROOT / 'data' / 'processed'

df = pd.read_parquet(PROC / 'dataset_analitico.parquet')
zonas = gpd.read_file(PROC / 'zonas_bogota.geojson')
print(df.shape, zonas.shape, zonas.crs)
df.head()

## Diccionario de columnas — `dataset_analitico.parquet`

Una fila por **`(cod_localidad × anio × tipo_delito)`** = 20 × 8 × 11 = **1.760 filas**.

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_localidad` | str(2) | Llave geográfica `01`–`20`. **Única llave de cruce.** |
| `localidad_nombre` | str | Referencia (nombre canónico desde la geometría). No usar para joins. |
| `anio` | int | 2018–2025. |
| `tipo_delito` | str | Prefijo SIEDCO: `H`, `HP`, `LP`, `HCE`, `DS`, `VI`, … |
| `tipo_delito_nombre` | str | Nombre legible del tipo de delito. |
| `conteo_siedco` | int | Casos SIEDCO de esa localidad-año-tipo (variable objetivo base). |
| `conteo_nuse` | int | Llamadas Línea 123 de la localidad-año (señal de contexto, taxonomía NUSE aparte; se repite por tipo). |
| `poblacion` | int | Población proyectada de la localidad ese año (SDP/DANE). |
| `ipm_nbi` | float | Incidencia de pobreza multidimensional (Censo DANE 2018, %). **Nulo en Sumapaz** (`20`). |
| `split` | str | `train` (2018–2024) / `test` (2025) — validación espacio-temporal sin fuga. |

In [ ]:
# Split espacio-temporal SIN fuga (ya viene marcado): entrenar ≤2024, evaluar 2025.
print(df['split'].value_counts())
train = df[df.split == 'train']
test = df[df.split == 'test']
print('train años:', sorted(train.anio.unique()))
print('test  años:', sorted(test.anio.unique()))

In [ ]:
# Ejemplo: unir el dataset con la geometría SIEMPRE por cod_localidad.
# (agregamos a nivel localidad-año para pintar un mapa de un tipo de delito)
hp_2025 = df[(df.tipo_delito == 'HP') & (df.anio == 2025)]
mapa = zonas.merge(hp_2025, on='cod_localidad', how='left')
mapa[['cod_localidad', 'localidad_nombre_x', 'conteo_siedco', 'ipm_nbi']].head()

In [ ]:
# Ojo con Sumapaz (20): ipm_nbi es NaN (no está en la Encuesta Multipropósito).
# Imputar o excluir según el modelo.
df[df.ipm_nbi.isna()][['cod_localidad', 'localidad_nombre', 'anio', 'ipm_nbi']].drop_duplicates('anio').head()